<a href="https://colab.research.google.com/github/kazi-amir/ml-learning/blob/main/Datathon-Practice/house-sale-price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Sale Price Prediction

In [117]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sbn

In [118]:
train_df = pd.read_csv('train.csv')
train_df.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [119]:
train_df.describe()

,Id,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,...,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SalePrice
count,1460.000000,1460.000000,1201.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1452.000000,1460.000000,...,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000,1460.000000
mean,730.500000,56.897260,70.049958,10516.828082,6.099315,5.575342,1971.267808,1984.865753,103.685262,443.639726,...,94.244521,46.660274,21.954110,3.409589,15.060959,2.758904,43.489041,6.321918,2007.815753,180921.195890
std,421.610009,42.300571,24.284752,9981.264932,1.382997,1.112799,30.202904,20.645407,181.066207,456.098091,...,125.338794,66.256028,61.119149,29.317331,55.757415,40.177307,496.123024,2.703626,1.328095,79442.502883
min,1.000000,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,34900.000000
25%,365.750000,20.000000,59.000000,7553.500000,5.000000,5.000000,1954.000000,1967.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.000000,2007.000000,129975.000000
50%,730.500000,50.000000,69.000000,9478.500000,6.000000,5.000000,1973.000000,1994.000000,0.000000,383.500000,...,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,163000.000000
75%,1095.250000,70.000000,80.000000,11601.500000,7.000000,6.000000,2000.000000,2004.000000,166.000000,712.250000,...,168.000000,68.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,214000.000000
max,1460.000000,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,5644.000000,...,857.000000,547.000000,552.000000,508.000000,480.000000,738.000000,15500.000000,12.000000,2010.000000,755000.000000


In [120]:
train_df.dropna(subset=['SalePrice'], inplace=True)
train_df.shape

(1460, 81)

In [121]:
for col in train_df.columns:
  if train_df[col].isna().sum() > 1000:
    train_df.drop(col, axis=1, inplace=True)

train_df.shape

(1460, 77)

In [122]:
train_in = train_df.drop('SalePrice', axis=1)
train_out = train_df['SalePrice']

In [123]:
train_in.shape, train_out.shape

((1460, 76), (1460,))

In [124]:
numeric_cols = train_in.select_dtypes(include=np.number).columns
categorical_cols = train_in.select_dtypes(exclude=np.number).columns

In [125]:
if 'Id' in numeric_cols:
    numeric_cols = numeric_cols.drop('Id')

In [126]:
numeric_cols = numeric_cols.to_list()
categorical_cols = categorical_cols.to_list()

In [127]:
numeric_cols

['MSSubClass',
 'LotFrontage',
 'LotArea',
 'OverallQual',
 'OverallCond',
 'YearBuilt',
 'YearRemodAdd',
 'MasVnrArea',
 'BsmtFinSF1',
 'BsmtFinSF2',
 'BsmtUnfSF',
 'TotalBsmtSF',
 '1stFlrSF',
 '2ndFlrSF',
 'LowQualFinSF',
 'GrLivArea',
 'BsmtFullBath',
 'BsmtHalfBath',
 'FullBath',
 'HalfBath',
 'BedroomAbvGr',
 'KitchenAbvGr',
 'TotRmsAbvGrd',
 'Fireplaces',
 'GarageYrBlt',
 'GarageCars',
 'GarageArea',
 'WoodDeckSF',
 'OpenPorchSF',
 'EnclosedPorch',
 '3SsnPorch',
 'ScreenPorch',
 'PoolArea',
 'MiscVal',
 'MoSold',
 'YrSold']

### Missing Values - Numeric

In [128]:
from sklearn.impute import SimpleImputer

In [129]:
imputer = SimpleImputer(strategy='mean')

In [130]:
imputer.fit(train_in[numeric_cols])

SimpleImputer()

In [131]:
train_in[numeric_cols] = imputer.transform(train_in[numeric_cols])

In [132]:
train_in.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,60.0,RL,65.0,8450.0,Pave,Reg,Lvl,AllPub,Inside,...,61.0,0.0,0.0,0.0,0.0,0.0,2.0,2008.0,WD,Normal
1,2,20.0,RL,80.0,9600.0,Pave,Reg,Lvl,AllPub,FR2,...,0.0,0.0,0.0,0.0,0.0,0.0,5.0,2007.0,WD,Normal
2,3,60.0,RL,68.0,11250.0,Pave,IR1,Lvl,AllPub,Inside,...,42.0,0.0,0.0,0.0,0.0,0.0,9.0,2008.0,WD,Normal
3,4,70.0,RL,60.0,9550.0,Pave,IR1,Lvl,AllPub,Corner,...,35.0,272.0,0.0,0.0,0.0,0.0,2.0,2006.0,WD,Abnorml
4,5,60.0,RL,84.0,14260.0,Pave,IR1,Lvl,AllPub,FR2,...,84.0,0.0,0.0,0.0,0.0,0.0,12.0,2008.0,WD,Normal


### Feature Scaling - MinMaxScaler

In [133]:
from sklearn.preprocessing import MinMaxScaler

In [134]:
scaler = MinMaxScaler()

In [135]:
scaler.fit(train_in[numeric_cols])

MinMaxScaler()

In [136]:
train_in[numeric_cols] = scaler.transform(train_in[numeric_cols])

In [137]:
train_in.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1,0.235294,RL,0.150685,0.033420,Pave,Reg,Lvl,AllPub,Inside,...,0.111517,0.000000,0.0,0.0,0.0,0.0,0.090909,0.50,WD,Normal
1,2,0.000000,RL,0.202055,0.038795,Pave,Reg,Lvl,AllPub,FR2,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.363636,0.25,WD,Normal
2,3,0.235294,RL,0.160959,0.046507,Pave,IR1,Lvl,AllPub,Inside,...,0.076782,0.000000,0.0,0.0,0.0,0.0,0.727273,0.50,WD,Normal
3,4,0.294118,RL,0.133562,0.038561,Pave,IR1,Lvl,AllPub,Corner,...,0.063985,0.492754,0.0,0.0,0.0,0.0,0.090909,0.00,WD,Abnorml
4,5,0.235294,RL,0.215753,0.060576,Pave,IR1,Lvl,AllPub,FR2,...,0.153565,0.000000,0.0,0.0,0.0,0.0,1.000000,0.50,WD,Normal


### Missing Values - Categoric

In [138]:
train_in[categorical_cols] = train_in[categorical_cols].fillna('Unknown')

### One Hot Encodig -Categoric

In [139]:
from sklearn.preprocessing import OneHotEncoder

In [140]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

In [141]:
categoric_no_na = train_df[categorical_cols].fillna('Unknown')

In [142]:
encoder.fit(categoric_no_na)

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [143]:
encoded_cols = list(encoder.get_feature_names_out(categorical_cols))

In [144]:
train_encoded = pd.DataFrame(
    encoder.transform(train_in[categorical_cols]),
    columns=encoded_cols,
    index=train_in.index
)

In [145]:
train_in = pd.concat([train_in, train_encoded], axis=1)

In [146]:
train_in.shape

(1460, 326)

In [147]:
for col in train_in.columns:
  if train_in[col].isna().sum() > 0:
    print(col, train_in[col].dtype)

In [149]:
X_train = train_in[numeric_cols + encoded_cols]
y_train = train_out

In [154]:
train_in.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,LotShape,LandContour,Utilities,LotConfig,...,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,1,0.235294,RL,0.150685,0.033420,Pave,Reg,Lvl,AllPub,Inside,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2,0.000000,RL,0.202055,0.038795,Pave,Reg,Lvl,AllPub,FR2,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,3,0.235294,RL,0.160959,0.046507,Pave,IR1,Lvl,AllPub,Inside,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,4,0.294118,RL,0.133562,0.038561,Pave,IR1,Lvl,AllPub,Corner,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,5,0.235294,RL,0.215753,0.060576,Pave,IR1,Lvl,AllPub,FR2,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [150]:
X_train.shape, y_train.shape

((1460, 286), (1460,))

## Train Validation split

In [152]:
from sklearn.model_selection import train_test_split

In [155]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
  )

In [156]:
X_train.shape, X_val.shape, y_train.shape, y_val.shape

((1168, 286), (292, 286), (1168,), (292,))

## Global Result Array

In [165]:
results = []

## Model Evaluation

In [162]:
def rmse(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()

    return np.sqrt(np.mean(np.square(t_vals - p_vals)))

In [163]:
def mae(targets, predictions):
    t_vals = np.asarray(targets).flatten()
    p_vals = np.asarray(predictions).flatten()

    return np.mean(np.abs(t_vals - p_vals))

In [169]:
def evaluate(model, targets, predictions):
  rmse_val = rmse(targets, predictions)
  mae_val = mae(targets, predictions)

  results.append({
      "model": model,
      "rmse": rmse_val,
      "mae": mae_val
    })

  print(f"RMSE: {rmse_val}, MAE: {mae_val}")

## Model Building

### Linear Regression

In [157]:
from sklearn.linear_model import LinearRegression

In [160]:
linear_model = LinearRegression()

In [161]:
linear_model.fit(X_train, y_train)

LinearRegression()

In [188]:
linear_preds = linear_model.predict(X_val)

In [189]:
evaluate("Linear Regression", y_val, linear_preds)

RMSE: 29604.04659942506, MAE: 18597.076451653353


### Decision Tree Regressor

In [171]:
from sklearn.tree import DecisionTreeRegressor

In [172]:
dt_model = DecisionTreeRegressor(random_state=42)

In [173]:
dt_model.fit(X_train, y_train)

DecisionTreeRegressor(random_state=42)

In [174]:
dt_preds = dt_model.predict(X_val)

In [175]:
evaluate("Decision Tree Regressor", y_val, dt_preds)

RMSE: 39983.10689652963, MAE: 26644.215753424658


### Decision Tree Regresson - max_depth (7)

In [179]:
dt_model_mxdpth = DecisionTreeRegressor(max_depth=7, random_state=42)

In [180]:
dt_model_mxdpth.fit(X_train, y_train)

DecisionTreeRegressor(max_depth=7, random_state=42)

In [181]:
dt_mxdpth_preds = dt_model_mxdpth.predict(X_val)

In [182]:
evaluate("Decision Tree Regressor - max_depth (7)", y_val, dt_mxdpth_preds)

RMSE: 36405.82537110885, MAE: 24577.889026712375


### Random Forrest Regressor

In [183]:
from sklearn.ensemble import RandomForestRegressor

In [184]:
rf_model = RandomForestRegressor(random_state=42)

In [185]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(random_state=42)

In [186]:
rf_preds = rf_model.predict(X_val)

In [187]:
evaluate("Random Forrest Regressor", y_val, rf_preds)

RMSE: 29075.05479965745, MAE: 17598.975616438354
